In [ ]:
#| default_exp calib_node

In [ ]:
#| hide
from nbdev.showdoc import *

# Calibration and Camera Info

> Publish camera calibration data alongside images. Load calibration from YAML and publish `CameraInfo` messages.

## Overview

A camera's raw image is distorted by its lens. To correct this (and to perform 3D reconstruction), you need the camera's **intrinsic parameters** (focal length, principal point) and **distortion coefficients**.

ROS2 represents this data in `sensor_msgs/CameraInfo`. Every camera node should publish a `CameraInfo` message synchronized with each image frame.

This notebook:

- Explains the `CameraInfo` message structure (K, D, R, P matrices)
- Shows how to load calibration from a YAML file
- Publishes `CameraInfo` alongside images
- Covers the `camera_calibration` workflow

## Imports

In [ ]:
#| export
#| hide
import sys
import yaml
import numpy as np
import rclpy
from rclpy.node import Node
from rclpy.executors import ExternalShutdownException
import cv2
from cv_bridge import CvBridge
from sensor_msgs.msg import Image, CameraInfo

## Import Camera from 02

In [ ]:
#| echo: false
from ros2_cam_nbdev.cam_pub import Camera

## Understanding CameraInfo

The `sensor_msgs/CameraInfo` message contains:

| Field | Description | Size |
|-------|-------------|------|
| `K` | Camera matrix (intrinsic) — focal length + principal point | 3×3 |
| `D` | Distortion coefficients | 5 or 8 |
| `R` | Rectification matrix | 3×3 |
| `P` | Projection matrix (for stereo) | 3×4 |
| `distortion_model` | Model type: `plumb_bob` (5 params) or `rational_polynomial` (8 params) | string |
| `width` / `height` | Image dimensions | int |

In [ ]:
#| export
def load_calibration(yaml_path):
    """Load camera calibration from a YAML file (camera_calibration format)."""
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)

    info = CameraInfo()
    info.width = data['image_width']
    info.height = data['image_height']
    info.distortion_model = data['distortion_model']
    info.d = data['distortion_coefficients']['data']
    info.k = data['camera_matrix']['data']
    info.r = data['rectification_matrix']['data']
    info.p = data['projection_matrix']['data']
    return info

## CalibNode

`CalibNode` publishes images and their corresponding `CameraInfo` messages on synchronized topics.

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `video_source` | string | `'0'` | Camera index or stream URL |
| `frame_rate` | float | `30.0` | Publishing frequency in FPS |
| `topic_name` | string | `'/camera/image_raw'` | Topic to publish images on |
| `calibration_file` | string | `''` | Path to calibration YAML (empty = publish empty CameraInfo) |
| `camera_name` | string | `'camera'` | Name used in CameraInfo header |

In [ ]:
#| export
class CalibNode(Node):
    """ROS2 camera publisher that also publishes synchronized CameraInfo."""
    def __init__(self):
        super().__init__('calib_node')

        # Declare parameters
        self.declare_parameter('video_source', '0')
        self.declare_parameter('frame_rate', 30.0)
        self.declare_parameter('topic_name', '/camera/image_raw')
        self.declare_parameter('calibration_file', '')
        self.declare_parameter('camera_name', 'camera')
        self.declare_parameter('video_fourcc', 'MJPG')
        self.declare_parameter('resolution_width', 640)
        self.declare_parameter('resolution_height', 480)

        # Read parameters
        source = self.get_parameter('video_source').value
        self.frame_rate = self.get_parameter('frame_rate').value
        self.topic_name = self.get_parameter('topic_name').value
        calib_file = self.get_parameter('calibration_file').value
        self.camera_name = self.get_parameter('camera_name').value
        fourcc = self.get_parameter('video_fourcc').value
        width = self.get_parameter('resolution_width').value
        height = self.get_parameter('resolution_height').value

        # Load calibration
        if calib_file:
            self.calib = load_calibration(calib_file)
            self.get_logger().info(f'Loaded calibration from {calib_file}')
        else:
            self.calib = CameraInfo()
            self.calib.width = width
            self.calib.height = height
            self.get_logger().info('No calibration file — publishing empty CameraInfo')

        # Initialize camera
        self.camera = Camera(source, fourcc, width, height)

        # Publishers
        self.img_pub = self.create_publisher(Image, self.topic_name, 20)
        info_topic = self.topic_name.replace('/image_raw', '/camera_info')
        self.info_pub = self.create_publisher(CameraInfo, info_topic, 20)

        # Timer
        period = 1.0 / self.frame_rate
        self.timer = self.create_timer(period, self._publish)

        # CvBridge
        self.bridge = CvBridge()
        self.frame_count = 0

        self.get_logger().info(
            f'Publishing image on "{self.topic_name}" and CameraInfo on "{info_topic}"'
        )

    def _publish(self):
        """Capture a frame and publish image + CameraInfo."""
        ret, frame = self.camera.read()
        if not ret:
            self.get_logger().warn('Failed to read frame')
            return

        stamp = self.get_clock().now().to_msg()

        # Image message
        img_msg = self.bridge.cv2_to_imgmsg(frame, encoding='bgr8')
        img_msg.header.stamp = stamp
        img_msg.header.frame_id = self.camera_name
        self.img_pub.publish(img_msg)

        # CameraInfo message (same timestamp and frame_id)
        info_msg = CameraInfo()
        info_msg.header.stamp = stamp
        info_msg.header.frame_id = self.camera_name
        info_msg.width = self.calib.width
        info_msg.height = self.calib.height
        info_msg.distortion_model = self.calib.distortion_model
        info_msg.d = self.calib.d
        info_msg.k = self.calib.k
        info_msg.r = self.calib.r
        info_msg.p = self.calib.p
        self.info_pub.publish(info_msg)

        if self.frame_count % 10 == 0:
            self.get_logger().info(f'Published frame {self.frame_count}')
        self.frame_count += 1

    def destroy_node(self):
        """Release the camera before destroying the node."""
        self.camera.release()
        super().destroy_node()

### show_doc

In [ ]:
show_doc(CalibNode)

## Entry Point

In [ ]:
#| export
def main(args=None):
    """Initialize ROS2, spin the CalibNode, and handle shutdown."""
    rclpy.init(args=args)
    node = CalibNode()

    try:
        rclpy.spin(node)
    except (KeyboardInterrupt, ExternalShutdownException):
        node.get_logger().info('Node stopped by user or external shutdown')
    finally:
        node.destroy_node()
        rclpy.try_shutdown()

In [ ]:
#| exporti
if not hasattr(sys, 'ps1'):
    if __name__ == "__main__":
        main()

## Calibration Workflow

### 1. Print a checkerboard

Use a pattern with known dimensions (e.g. 9×6 inner corners, 25mm square size).

### 2. Capture images

```sh
ros2 run camera_calibration cameracalibrator \
    --size 9x6 --square 0.025 \
    image:=/camera/image_raw
```

Move the checkerboard in front of the camera at different angles until the calibration bar fills up.

### 3. Calibrate

Click **Calibrate** in the GUI. The tool computes K, D, R, P matrices and saves them to a YAML file.

### 4. Use the calibration

```sh
ros2 run ros2_cam_nbdev calib_node --ros-args \
    -p calibration_file:=/path/to/calibration.yaml
```

## Usage

First, export the notebook to generate the Python module:

```bash
uv run nbdev-export
```

This creates `ros2_cam_nbdev/calib_node.py`.

### Method 1 — Run directly with Python

The fastest way to test. No workspace or compilation needed.

**Terminal 1** — run the node without calibration:

```bash
uv run python -m ros2_cam_nbdev.calib_node
```

Or with a calibration file:

```bash
uv run python -m ros2_cam_nbdev.calib_node --ros-args \
    -p calibration_file:=/path/to/calibration.yaml
```

**Terminal 2** — check the topics:

```bash
ros2 topic hz /camera/image_raw
```

```bash
ros2 topic echo /camera/camera_info
```

See `08_visualization` for tools to view the images.

### Method 2 — Full ROS2 workflow

The standard ROS2 way: create a workspace, package, compile, and run.

**Step 1** — Source ROS2:

```bash
source /opt/ros/$ROS_DISTRO/setup.bash
```

> Use `setup.zsh` if your shell is zsh.

**Step 2** — Create the workspace:

```bash
zsh scripts/generate_ros2_ws.zsh new_ros2_ws
```

**Step 3** — Create the package:

```bash
zsh scripts/mk_ros2_pkg.sh new_ros2_ws my_camera_pkg camera_opencv.txt
```

> This creates `new_ros2_ws/src/my_camera_pkg/` with ROS2 dependencies
> read from `scripts/pkg_dependencies/camera_opencv.txt`.

**Step 4** — Copy the node into the package:

```bash
cp ros2_cam_nbdev/calib_node.py new_ros2_ws/src/my_camera_pkg/my_camera_pkg/
```

**Step 5** — Compile:

```bash
bash scripts/compile_pkg.sh new_ros2_ws my_camera_pkg
```

> `compile_pkg.sh` generates `setup.py` automatically (entry points are
> discovered from any file with `def main(args=None):`) and runs `colcon build`.

**Step 6** — Source the workspace overlay:

```bash
source new_ros2_ws/install/setup.bash
```

> Use `setup.zsh` if your shell is zsh.

**Step 7** — Run the node:

```bash
ros2 run my_camera_pkg calib_node
```

Or with a calibration file:

```bash
ros2 run my_camera_pkg calib_node --ros-args \
    -p calibration_file:=/path/to/calibration.yaml
```

**Terminal 2** — verify and visualize:

```bash
ros2 topic hz /camera/image_raw
```

See `08_visualization` for tools to view the images.

## Visualize

With the node running, open a second terminal and visualize the images:

```sh
ros2 run rqt_image_view rqt_image_view
```

Select `/camera/image_raw` from the dropdown.

You can also inspect the calibration info:

```sh
ros2 topic echo /camera/camera_info
```

For more visualization options (rviz2, Foxglove), see `08_visualization`.

## Next Steps

We've covered the core camera node patterns. Next: `07_simulation` for running without hardware, `10_existing_drivers` for production-ready alternatives, and `11_annex` for WSL tips and common pitfalls.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()